In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH="telecom_guide.pdf"

loader=PyPDFLoader(PDF_PATH)
pages=loader.load()

print(f"Loaded {len(pages)} from pdf.")
print("\n------first page preview(first 500 chars)------")
print(pages[0].page_content[:500])

Loaded 9 from pdf.

------first page preview(first 500 chars)------
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [13]:
print(pages[1].page_content[:500])

Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, an


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n","\n","."," "]
)
chunks=splitter.split_documents(pages)
len(chunks)

37

In [15]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store=Chroma.from_documents(chunks,embeddings)

print(f"vector store ready.{vector_store._collection.count()} vector stored")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2238.40it/s]


vector store ready.37 vector stored


In [17]:
retriever=vector_store.as_retriever(search_kwargs={"k":3})

test_query="what is VoLTE and how does it improve call quality??"
retrieved=retriever.invoke(test_query)

for chunk,doc in enumerate(retrieved,1):
    print(f"----chunk{chunk}----")
    print(doc.page_content[:300])
    print()

----chunk1----
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

----chunk2----
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt

----chunk3----
prioritised over general data traffic. This prevents voice quality degradation during periods of network congestion.
Without QoS, voice packets would compete with video streaming and file downloads, causing jitter and packet
loss.
Fallback Behaviour: If a VoLTE call cannot be established  - for exam



In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

def format_docs(docs):
    return "\n\n--\n\n".join(
        doc.page_content for doc in docs
    )

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    reasoning_format="parsed"
)

chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled")

RAG chain assembled


In [20]:
question="how does internation roaming work and what charges should i expect?"

print(f"Q:{question}\n")
print("A:",chain.invoke(question))

Q:how does internation roaming work and what charges should i expect?

A: **How international roaming works**

1. **Connection to a partner network** – When you leave the coverage area of your home network, your phone automatically looks for a local (visited‑country) network that has a roaming agreement with your carrier.  
2. **Authentication** – The visited network authenticates you using an inter‑operator signalling protocol (SS7 or Diameter).  
3. **Home‑network authorization** – Your home network receives the authentication request, checks that your subscription is valid, and authorises the requested services (voice, data, SMS).  
4. **Service delivery** – Once authorised, all of your traffic (voice calls, mobile data, SMS) is routed through the visited network but billed by your home carrier.

**Charges you can expect**

| Item | What the guide says |
|------|----------------------|
| **Base roaming rates** | You are charged for data, voice and SMS according to the roaming tariff